In [1]:
from utils import get_stage_packages
get_stage_packages()
! pip install -r pip-requirements.txt

Processing /tmp/dist/ml_utils-0.0.1-py3-none-any.whl (from -r pip-requirements.txt (line 1))
  Using cached snowflake_ml_python-1.30.0-py3-none-any.whl.metadata (113 kB)
  Using cached importlib_resources-6.5.2-py3-none-any.whl.metadata (3.9 kB)
  Using cached pytimeparse-1.1.8-py2.py3-none-any.whl.metadata (3.4 kB)
  Using cached retrying-1.4.2-py3-none-any.whl.metadata (5.5 kB)
  Using cached sqlparse-0.5.5-py3-none-any.whl.metadata (4.7 kB)
  Using cached xgboost-3.2.0-py3-none-macosx_12_0_arm64.whl.metadata (2.1 kB)
  Using cached slicer-0.0.8-py3-none-any.whl.metadata (4.0 kB)
INFO: pip is looking at multiple versions of boto3 to determine which version is compatible with other requirements. This could take a while.
  Using cached boto3-1.42.67-py3-none-any.whl.metadata (6.7 kB)
  Using cached boto3-1.42.66-py3-none-any.whl.metadata (6.7 kB)
  Using cached boto3-1.42.65-py3-none-any.whl.metadata (6.7 kB)
INFO: pip is still looking at multiple versions of boto3 to determine which v

In [2]:
from ml_utils.utils import version_data

from sklearn.linear_model import LinearRegression

from snowflake.snowpark.session import Session
from snowflake.ml.feature_store import FeatureStore, CreationMode
from snowflake.ml.registry import Registry
from snowflake.ml.dataset import Dataset

from ml_utils.snowflake_env import get_session

/Users/aferas/repos/ml-pipelines/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Create session
session = get_session()


'"ML_DEV_DB"'

In [ ]:
# Get Data

fs = FeatureStore(
    session=session,
    database=session.get_current_database(),
    name=session.get_current_schema(),
    default_warehouse=session.get_current_warehouse(),
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST
)

fv = fs.get_feature_view("EXAMPLE_FEATURES",version="3")
df = fs.read_feature_view(fv).sample(n=1000)

In [ ]:
# Save as dataset
ds_name = "TRAIN_DATASET"
df = df.select("INCOME","MORTGAGERESPONSE")
ds = Dataset.create(session=session, name=ds_name, exist_ok=True)
version_name = version_data(df)
ds_version = ds.create_version(version=version_name, input_dataframe=df, label_cols=["MORTGAGERESPONSE"])

In [ ]:
# Train Model
df = df.to_pandas()

X = df.drop(columns=["MORTGAGERESPONSE"]).fillna(0)
y = df[["MORTGAGERESPONSE"]]

lr = LinearRegression()
lr.fit(X, y)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [ ]:
# Register model
reg = Registry(session=session)

model_name = "MODEL_EX1"
mv = reg.log_model(
    model=lr, 
    model_name=model_name, 
    sample_input_data=ds_version.read.to_snowpark_dataframe().drop("MORTGAGERESPONSE").fillna(0).limit(100), 
    target_platforms=["WAREHOUSE", "SNOWPARK_CONTAINER_SERVICES"] )

reg.show_models()


Logging model: creating model manifest...:  33%|███▎      | 2/6 [00:00<00:01,  3.89it/s]  

/Users/aferas/repos/ml-pipelines/.venv/lib/python3.10/site-packages/snowflake/ml/registry/_manager/model_parameter_reconciler.py:72: UserWarning: `relax_version` is not set and therefore defaulted to True. Dependency version constraints relaxed from ==x.y.z to >=x.y, <(x+1). To use specific dependency versions for compatibility, reproducibility, etc., set `options={'relax_version': False}` when logging the model.
  reconciled_options = self._reconcile_relax_version(reconciled_options, reconciled_target_platforms)
/Users/aferas/repos/ml-pipelines/.venv/lib/python3.10/site-packages/snowflake/ml/model/model_signature.py:74: UserWarning: The sample input has 100 rows. Using the first 100 rows to define the inputs and outputs of the model and the data types of each. Use `signatures` parameter to specify model inputs and outputs manually if the automatic inference is not correct.
  warnings.warn(


Model logged successfully.: 100%|██████████| 6/6 [01:27<00:00, 14.61s/it]                          


,created_on,name,model_type,database_name,schema_name,comment,owner,default_version_name,versions,aliases
0,2026-03-09 18:59:59.887000-07:00,JOURNEY_CONVERSION_MODEL_SNOWXGB,USER_MODEL,ML_DEV_DB,ML_DEV_SCHEMA,None,ML_DEV_ROLE,RED_FIREANT_4,"[""BAD_DUCK_3"",""LAZY_GIBBON_1"",""RED_FIREANT_4"",...","{""DEFAULT"":""RED_FIREANT_4"",""FIRST"":""RED_FIREAN..."
1,2026-03-18 09:52:22.305000-07:00,MODEL_EX1,USER_MODEL,ML_DEV_DB,ML_DEV_SCHEMA,None,ML_DEV_ROLE,HAPPY_BULLFROG_4,"[""HAPPY_BULLFROG_4""]","{""DEFAULT"":""HAPPY_BULLFROG_4"",""FIRST"":""HAPPY_B..."


In [ ]:
# promote to default
base_model = reg.get_model(model_name)
base_model.default = mv

: 